# Data Cleaning and Preprocessing in Python

This notebook teaches data cleaning and preprocessing using the same CSV from the main data science course:

`student_performance_test_dataset.csv`

The lesson intentionally makes a messy copy of the dataset so students can practice real data preparation skills:

1. Loading CSV data
2. Understanding data quality problems
3. Adding missing values for practice
4. Finding duplicates
5. Fixing inconsistent categories
6. Fixing impossible values and outliers
7. Handling missing numeric and categorical data
8. Feature engineering
9. One-hot encoding
10. Scaling numeric features
11. Train-test split
12. Full preprocessing pipelines
13. Creating a model-ready dataset
14. Training a simple model after preprocessing
15. Saving cleaned and processed data

Goal: learn how raw data becomes clean, model-ready data.

## 1. Setup

Run this first. If a package is missing, the setup cell tries to install it into the current notebook kernel.

In [3]:
# Package check for classroom machines.
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}


missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    try:
        from IPython import get_ipython

        ipython = get_ipython()
        if ipython is not None:
            ipython.run_line_magic("pip", "install " + " ".join(missing_packages))
        else:
            subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    except Exception as error:
        print("Automatic installation did not work in this Python environment.")
        print("Please run this in a notebook cell instead:")
        print("%pip install numpy pandas matplotlib scikit-learn")
        raise error
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [26]:
# Main libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn preprocessing and modeling tools
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# pd.set_option("display.max_columns", 80)
# pd.set_option("display.width", 120)

RANDOM_SEED = 50
np.random.seed(RANDOM_SEED)

print(np.random.random(RANDOM_SEED))


# 100% > 80% training + 20% testing

[0.49460165 0.2280831  0.25547392 0.39632991 0.3773151  0.99657423
 0.4081972  0.77189399 0.76053669 0.31000935 0.3465412  0.35176482
 0.14546686 0.97266468 0.90917844 0.5599571  0.31359075 0.88820004
 0.67457307 0.39108745 0.50718412 0.5241035  0.92800093 0.57137307
 0.66833757 0.05225869 0.3270573  0.05640164 0.17982769 0.92593317
 0.93801522 0.71409271 0.73268761 0.46174768 0.93132927 0.40642024
 0.68320577 0.64991587 0.59876518 0.22203939 0.68235717 0.8780563
 0.79671726 0.43200225 0.91787822 0.78183368 0.72575028 0.12485469
 0.91630845 0.38771099]


In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name": np.random.choice(["Alice", "Bob", "Charlie", "David", "Emma"], size=10),
    "age": np.random.randint(18, 60, size=10),
    "salary": np.random.randint(30000, 150000, size=10),
    "city": np.random.choice(["Mumbai", "Delhi", "Pune", "Bangalore"], size=10),
    "is_active": np.random.choice([True, False], size=10)
})

print(df.info())
print(df.head())
print(df.describe())


<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   name       10 non-null     str  
 1   age        10 non-null     int64
 2   salary     10 non-null     int64
 3   city       10 non-null     str  
 4   is_active  10 non-null     bool 
dtypes: bool(1), int64(2), str(2)
memory usage: 462.0 bytes
None
      name  age  salary       city  is_active
0      Bob   34  100118  Bangalore       True
1      Bob   44   81904     Mumbai      False
2  Charlie   22  125739      Delhi      False
3     Emma   19   57982       Pune       True
4      Bob   55   83812       Pune      False
             age         salary
count  10.000000      10.000000
mean   36.800000   87274.500000
std    14.823405   23140.287938
min    19.000000   48458.000000
25%    24.250000   76141.750000
50%    33.000000   87916.000000
75%    50.750000  102955.250000
max    58.000000  125739.000000


In [29]:
df.head()

,name,age,salary,city,is_active
0,Bob,34,100118,Bangalore,True
1,Bob,44,81904,Mumbai,False
2,Charlie,22,125739,Delhi,False
3,Emma,19,57982,Pune,True
4,Bob,55,83812,Pune,False


In [30]:

RANDOM_SEED = 100
np.random.seed(RANDOM_SEED)
np.random.random(10)



array([0.54340494, 0.27836939, 0.42451759, 0.84477613, 0.00471886,
       0.12156912, 0.67074908, 0.82585276, 0.13670659, 0.57509333])

## 2. Load the Same CSV Dataset

This notebook expects the CSV file to be in the same folder as the notebook.

If the file does not exist, run the first notebook once to create it.

In [31]:
csv_file = "student_performance_test_dataset.csv"
raw_data = pd.read_csv(csv_file)


print("Rows and columns:", raw_data.shape)
raw_data.head()

Rows and columns: (505, 10)


,student_id,study_hours,attendance_rate,previous_score,sleep_hours,internet_access,parent_education,extra_classes,final_score,passed
0,1,6.5,89.1,81.6,7.7,Yes,High School,Yes,82.2,1
1,2,5.2,100.0,74.9,6.2,Yes,High School,No,78.2,1
2,3,6.8,61.2,62.8,5.9,Yes,High School,No,67.6,0
3,4,8.5,84.8,52.9,6.8,NaN,Bachelor,Yes,83.3,1
4,5,NaN,70.2,71.8,6.6,Yes,High School,No,69.1,0


## 3. First Data Quality Check

Before cleaning, always inspect the data. This tells us what type of cleaning is needed.

In [32]:
# Column names, data types, and non-missing counts
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        505 non-null    int64  
 1   study_hours       493 non-null    float64
 2   attendance_rate   493 non-null    float64
 3   previous_score    505 non-null    float64
 4   sleep_hours       505 non-null    float64
 5   internet_access   493 non-null    str    
 6   parent_education  505 non-null    str    
 7   extra_classes     505 non-null    str    
 8   final_score       505 non-null    float64
 9   passed            505 non-null    int64  
dtypes: float64(5), int64(2), str(3)
memory usage: 39.6 KB


In [33]:
# Summary of numeric columns
raw_data.describe()

,student_id,study_hours,attendance_rate,previous_score,sleep_hours,final_score,passed
count,505.000000,493.000000,493.000000,505.000000,505.000000,505.000000,505.000000
mean,250.142574,5.544016,78.259838,63.582772,6.835446,70.691881,0.526733
std,144.377054,1.955841,11.426800,14.154383,1.081221,10.503788,0.499780
min,1.000000,0.500000,45.600000,21.500000,3.600000,39.000000,0.000000
25%,125.000000,4.100000,71.100000,53.600000,6.100000,63.700000,0.000000
50%,250.000000,5.500000,78.300000,63.700000,6.800000,70.400000,1.000000
75%,375.000000,6.800000,85.800000,72.600000,7.600000,77.800000,1.000000
max,500.000000,12.000000,100.000000,98.400000,10.000000,100.000000,1.000000


In [39]:
# Missing values per column
raw_data.isna().sum().sort_values(ascending=False)

study_hours         12
attendance_rate     12
internet_access     12
student_id           0
previous_score       0
sleep_hours          0
parent_education     0
extra_classes        0
final_score          0
passed               0
dtype: int64

In [40]:
# Duplicate rows
print("Duplicate rows:", raw_data.duplicated().sum())

Duplicate rows: 5


## 4. Create a Messy Practice Dataset

Real-world data is rarely clean. We will intentionally add common problems:

- More missing values
- Duplicates
- Inconsistent category spelling
- Impossible numeric values
- Outliers
- Text spaces around category values

This gives students a realistic cleaning exercise.

In [41]:
messy_data = raw_data.copy()
messy_data.head()

for column, count in {
    "study_hours": 25,
    "attendance_rate": 20,
    "previous_score": 15,
    "sleep_hours": 12,
    "internet_access": 18,
    "parent_education": 10,
    "extra_classes": 14,
}.items():
    rows = np.random.choice(messy_data.index, size=count, replace=False)
    messy_data.loc[rows, column] = np.nan


messy_data.head()

,student_id,study_hours,attendance_rate,previous_score,sleep_hours,internet_access,parent_education,extra_classes,final_score,passed
0,1,6.5,89.1,81.6,7.7,Yes,High School,Yes,82.2,1
1,2,5.2,100.0,74.9,6.2,Yes,High School,No,78.2,1
2,3,6.8,61.2,62.8,5.9,Yes,High School,No,67.6,0
3,4,8.5,84.8,52.9,6.8,NaN,Bachelor,Yes,83.3,1
4,5,NaN,70.2,71.8,6.6,Yes,High School,No,69.1,0


In [51]:
messy_data.loc[[4],['final_score']]


,final_score
4,69.1


In [55]:
!pip install tabulate 


In [57]:
messy_data.to_csv('test.csv')

In [52]:
# Messy data creation


messy_data = raw_data.copy()

# Add extra missing values in several columns.
for column, count in {
    "study_hours": 25,
    "attendance_rate": 20,
    "previous_score": 15,
    "sleep_hours": 12,
    "internet_access": 18,
    "parent_education": 10,
    "extra_classes": 14,
}.items():
    rows = np.random.choice(messy_data.index, size=count, replace=False)
    messy_data.loc[rows, column] = np.nan

# Add inconsistent category values.
messy_data.loc[np.random.choice(messy_data.index, 8, replace=False), "internet_access"] = " yes "
messy_data.loc[np.random.choice(messy_data.index, 7, replace=False), "internet_access"] = "NO"
messy_data.loc[np.random.choice(messy_data.index, 6, replace=False), "extra_classes"] = " yes"
messy_data.loc[np.random.choice(messy_data.index, 6, replace=False), "extra_classes"] = "No "
messy_data.loc[np.random.choice(messy_data.index, 5, replace=False), "parent_education"] = "bachelor"
messy_data.loc[np.random.choice(messy_data.index, 5, replace=False), "parent_education"] = "Masters"

# Add impossible values and outliers.
messy_data.loc[np.random.choice(messy_data.index, 4, replace=False), "study_hours"] = -2
messy_data.loc[np.random.choice(messy_data.index, 4, replace=False), "study_hours"] = 30
messy_data.loc[np.random.choice(messy_data.index, 4, replace=False), "attendance_rate"] = 150
messy_data.loc[np.random.choice(messy_data.index, 4, replace=False), "previous_score"] = -10
messy_data.loc[np.random.choice(messy_data.index, 4, replace=False), "sleep_hours"] = 18

# Add duplicate rows.
messy_data = pd.concat([messy_data, messy_data.sample(10, random_state=RANDOM_SEED)], ignore_index=True)

print("Original shape:", raw_data.shape)
print("Messy shape:", messy_data.shape)
messy_data.head(10)

Original shape: (505, 10)
Messy shape: (515, 10)


,student_id,study_hours,attendance_rate,previous_score,sleep_hours,internet_access,parent_education,extra_classes,final_score,passed
0,1,6.5,89.1,81.6,7.7,Yes,High School,Yes,82.2,1
1,2,5.2,100.0,74.9,6.2,Yes,High School,No,78.2,1
2,3,6.8,61.2,62.8,5.9,Yes,High School,No,67.6,0
3,4,8.5,NaN,52.9,6.8,NaN,Bachelor,NaN,83.3,1
4,5,NaN,70.2,71.8,6.6,Yes,High School,No,69.1,0
5,6,5.0,72.2,67.5,6.3,No,High School,No,69.0,0
6,7,8.7,70.9,74.5,7.6,Yes,Bachelor,Yes,79.6,1
7,8,7.0,NaN,70.9,7.9,No,PhD,No,68.4,0
8,9,4.6,78.6,76.7,6.9,Yes,Bachelor,Yes,70.8,1
9,10,6.6,68.0,54.5,NaN,yes,High School,No,73.9,1


In [58]:
# Save the messy version so students can practice loading a dirty file.
messy_data.to_csv("student_performance_messy_dataset.csv", index=False)
print("Saved: student_performance_messy_dataset.csv")

Saved: student_performance_messy_dataset.csv


In [59]:
messy_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 515 entries, 0 to 514
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        515 non-null    int64  
 1   study_hours       478 non-null    float64
 2   attendance_rate   482 non-null    float64
 3   previous_score    499 non-null    float64
 4   sleep_hours       502 non-null    float64
 5   internet_access   487 non-null    str    
 6   parent_education  506 non-null    str    
 7   extra_classes     501 non-null    str    
 8   final_score       515 non-null    float64
 9   passed            515 non-null    int64  
dtypes: float64(5), int64(2), str(3)
memory usage: 40.4 KB


/var/folders/mn/glzj3r6d7d71hjy8fz5lq5lm0000gn/T/ipykernel_17244/1555002680.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  messy_data['parent_education'].mask(messy_data['parent_education'].isnull(), "Unknown", inplace=True).unique()


<StringArray>
['High School', 'Bachelor', 'bachelor', 'Master', 'PhD', 'Unknown', 'Masters']
Length: 7, dtype: str

## 5. Build a Data Quality Report

A data quality report gives a quick view of problems in each column.

In [66]:
messy_data.iloc[messy_data["sleep_hours"].isna()]

,student_id,study_hours,attendance_rate,previous_score,sleep_hours,internet_access,parent_education,extra_classes,final_score,passed
9,10,6.6,68.0,54.5,NaN,yes,High School,No,73.9,1
18,19,3.7,150.0,57.5,NaN,Yes,NaN,No,47.8,0
21,22,5.0,84.5,54.0,NaN,No,Bachelor,No,66.8,0
66,67,5.4,53.5,43.9,NaN,Yes,High School,No,60.1,0
118,119,7.8,83.8,64.8,NaN,Yes,Master,No,79.0,1
177,178,8.4,99.0,60.6,NaN,Yes,Master,NaN,79.1,1
197,198,5.8,63.0,69.7,NaN,Yes,High School,Yes,60.9,0
222,223,6.9,61.8,70.9,NaN,Yes,High School,No,67.8,0
265,266,3.0,69.6,42.2,NaN,Yes,High School,No,57.5,0
374,375,9.8,77.6,82.6,NaN,Yes,Bachelor,No,93.0,1


In [72]:
messy_data.isna().sum()

student_id           0
study_hours         37
attendance_rate     33
previous_score      16
sleep_hours         13
internet_access     28
parent_education     9
extra_classes       14
final_score          0
passed               0
dtype: int64

In [81]:
messy_data.nunique(dropna=True)

student_id          500
study_hours          88
attendance_rate     276
previous_score      316
sleep_hours          59
internet_access       4
parent_education      6
extra_classes         4
final_score         294
passed                2
dtype: int64

In [83]:
messy_data['extra_classes'].unique()

<StringArray>
['Yes', 'No', nan, 'No ', ' yes']
Length: 5, dtype: str

In [82]:
def data_quality_report(df):
    """Return a compact quality report for every column in a DataFrame."""
    report = pd.DataFrame(
        {
            "dtype": df.dtypes,
            "missing_count": df.isna().sum(),
            "missing_percent": (df.isna().mean() * 100).round(2),
            "unique_values": df.nunique(dropna=True),
        }
    )
    return report




data_quality_report(messy_data)

,dtype,missing_count,missing_percent,unique_values
student_id,int64,0,0.00,500
study_hours,float64,37,7.18,88
attendance_rate,float64,33,6.41,276
previous_score,float64,16,3.11,316
sleep_hours,float64,13,2.52,59
internet_access,str,28,5.44,4
parent_education,str,9,1.75,6
extra_classes,str,14,2.72,4
final_score,float64,0,0.00,294
passed,int64,0,0.00,2


In [87]:
# Inspect unique values in categorical columns.
for column in ["internet_access", "parent_education", "extra_classes"]:
    print(f'Column : {column} , Data : {messy_data[column].unique()}')

Column : internet_access , Data : <StringArray>
['Yes', nan, 'No', ' yes ', 'NO']
Length: 5, dtype: str
Column : parent_education , Data : <StringArray>
['High School', 'Bachelor', 'PhD', nan, 'Master', 'bachelor', 'Masters']
Length: 7, dtype: str
Column : extra_classes , Data : <StringArray>
['Yes', 'No', nan, 'No ', ' yes']
Length: 5, dtype: str


## 6. Manual Cleaning Step by Step

Manual cleaning is useful for learning and for exploratory analysis.

Later, we will build a scikit-learn pipeline so the same logic can be applied safely to training and test data.

In [100]:
messy_data[messy_data['student_id'].duplicated()]

,student_id,study_hours,attendance_rate,previous_score,sleep_hours,internet_access,parent_education,extra_classes,final_score,passed
500,362,8.6,59.8,70.1,7.7,Yes,High School,Yes,71.1,1
501,74,8.6,75.4,44.0,5.3,No,High School,Yes,59.0,0
502,375,9.8,77.6,82.6,8.2,Yes,Bachelor,No,93.0,1
503,156,4.1,78.7,76.1,6.5,NaN,Master,No,72.6,1
504,105,5.2,83.0,77.5,6.3,Yes,Bachelor,No,75.8,1
505,384,4.4,NaN,53.5,NaN,Yes,PhD,Yes,81.6,1
506,228,3.3,82.2,59.9,7.7,Yes,Master,No,74.4,1
507,70,4.2,73.8,66.9,6.2,Yes,High School,No,61.1,0
508,32,9.2,58.1,38.6,4.9,Yes,Master,No,66.1,0
509,334,4.1,68.8,NaN,7.1,No,Bachelor,No,78.4,1


In [101]:
clean_data = messy_data.copy()

# Remove exact duplicate rows.
before = clean_data.shape[0]


clean_data = clean_data.drop_duplicates()
after = clean_data.shape[0]

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", after)
print("Rows removed:", before - after)

Rows before duplicate removal: 515
Rows after duplicate removal: 502
Rows removed: 13


### Clean Categorical Text

Category problems often come from inconsistent typing, capitalization, or spaces.

Examples:

- `Yes`, ` yes `, and `YES` should all become `Yes`.
- `Masters` should become `Master`.

In [108]:
messy_data['parent_education'].unique()

<StringArray>
['High School', 'Bachelor', 'PhD', nan, 'Master', 'bachelor', 'Masters']
Length: 7, dtype: str

In [109]:
def clean_yes_no(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip()
    if value in ['yes', 'y', 'Yes']:
        return "Yes"
    if value in ['No','N','no']:
        return "No"

   



def clean_parent_education(value):
    if pd.isna(value):
        return np.nan
    

    value = str(value).strip().lower()
    
    mapping = {
        'high school' : 'High School',
        'bachelor' : 'Bachelor',
        'phd' : 'PhD', 
        'master' : 'Masters',
        'bachelor': 'Bachelor',
        'masters' : 'Masters'
    }
    return mapping.get(value, np.nan)






clean_data["internet_access"] = clean_data["internet_access"].apply(clean_yes_no)
clean_data["extra_classes"] = clean_data["extra_classes"].apply(clean_yes_no)
clean_data["parent_education"] = clean_data["parent_education"].apply(clean_parent_education)





In [111]:
for column in ["internet_access", "parent_education", "extra_classes"]:
    print(f'Column : {column} , Data : {clean_data[column].unique()}')

Column : internet_access , Data : <StringArray>
['Yes', nan, 'No']
Length: 3, dtype: str
Column : parent_education , Data : <StringArray>
['High School', 'Bachelor', 'PhD', nan, 'Masters']
Length: 5, dtype: str
Column : extra_classes , Data : <StringArray>
['Yes', 'No', nan]
Length: 3, dtype: str


### Fix Impossible Numeric Values

Some values are not realistic:

- Study hours cannot be negative.
- Attendance rate should be between 0 and 100.
- Previous score should be between 0 and 100.
- Sleep hours should be in a realistic range.

For teaching, we convert impossible values to missing values, then impute them later.

In [ ]:
clean_data.loc[(clean_data["study_hours"] < 0) | (clean_data["study_hours"] > 16), "study_hours"] = np.nan
clean_data.loc[(clean_data["attendance_rate"] < 0) | (clean_data["attendance_rate"] > 100), "attendance_rate"] = np.nan
clean_data.loc[(clean_data["previous_score"] < 0) | (clean_data["previous_score"] > 100), "previous_score"] = np.nan
clean_data.loc[(clean_data["sleep_hours"] < 0) | (clean_data["sleep_hours"] > 12), "sleep_hours"] = np.nan

clean_data[["study_hours", "attendance_rate", "previous_score", "sleep_hours"]].describe()

### Missing Value Strategies

Common choices:

- Drop rows: simple but can lose too much data.
- Fill numeric columns with mean or median.
- Fill categorical columns with the most common value.
- Use an advanced model-based imputer for complex projects.

Median is often safer than mean when outliers exist.

In [ ]:
numeric_columns = ["study_hours", "attendance_rate", "previous_score", "sleep_hours"]
categorical_columns = ["internet_access", "parent_education", "extra_classes"]

# Fill numeric missing values with the median.
for column in numeric_columns:
    clean_data[column] = clean_data[column].fillna(clean_data[column].median())

# Fill categorical missing values with the most frequent value.
for column in categorical_columns:
    clean_data[column] = clean_data[column].fillna(clean_data[column].mode()[0])

clean_data.isna().sum()

## 7. Outlier Detection with the IQR Method

The Interquartile Range method uses the middle 50% of values.

- Q1: 25th percentile
- Q3: 75th percentile
- IQR: Q3 - Q1
- Lower limit: Q1 - 1.5 * IQR
- Upper limit: Q3 + 1.5 * IQR

Values outside the limits may be outliers.

In [ ]:
def iqr_limits(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for column in numeric_columns:
    lower, upper = iqr_limits(clean_data[column])
    outlier_count = ((clean_data[column] < lower) | (clean_data[column] > upper)).sum()
    print(f"{column}: lower={lower:.2f}, upper={upper:.2f}, outliers={outlier_count}")

In [ ]:
# Optional approach: cap outliers instead of deleting them.
# This is also called winsorization.
capped_data = clean_data.copy()

for column in numeric_columns:
    lower, upper = iqr_limits(capped_data[column])
    capped_data[column] = capped_data[column].clip(lower=lower, upper=upper)

capped_data[numeric_columns].describe()

## 8. Feature Engineering

Feature engineering creates new useful columns from existing data.

In [ ]:
processed_base = capped_data.copy()

# Combined behavior feature: study hours adjusted by attendance.
processed_base["study_attendance_index"] = processed_base["study_hours"] * (processed_base["attendance_rate"] / 100)

# Improvement opportunity: how far the previous score is from 100.
processed_base["score_gap"] = 100 - processed_base["previous_score"]

# Binary feature: did the student sleep at least 7 hours?
processed_base["healthy_sleep"] = np.where(processed_base["sleep_hours"] >= 7, "Yes", "No")

processed_base[["study_hours", "attendance_rate", "study_attendance_index", "previous_score", "score_gap", "healthy_sleep"]].head()

## 9. One-Hot Encoding by Hand with pandas

Machine learning models need numbers, not text categories.

One-hot encoding creates one column per category.

Example: `internet_access` becomes columns like `internet_access_Yes` and `internet_access_No`.

In [ ]:
categorical_for_encoding = ["internet_access", "parent_education", "extra_classes", "healthy_sleep"]

one_hot_demo = pd.get_dummies(
    processed_base[categorical_for_encoding],
    columns=categorical_for_encoding,
    drop_first=False,
    dtype=int,
)

one_hot_demo.head()

### Avoid the Dummy Variable Trap

For some models, one category per feature can be dropped because it is represented by all zeros in the remaining columns.

`drop_first=True` does that. Tree models usually do not require it, but linear models often benefit from it.

In [ ]:
one_hot_drop_first_demo = pd.get_dummies(
    processed_base[categorical_for_encoding],
    columns=categorical_for_encoding,
    drop_first=True,
    dtype=int,
)

one_hot_drop_first_demo.head()

## 10. Ordinal Encoding

Ordinal encoding is useful when categories have a meaningful order.

`parent_education` has an order:

`High School < Bachelor < Master < PhD`

Important: do not use ordinal encoding for categories without real order, such as city names.

In [ ]:
education_order = ["High School", "Bachelor", "Master", "PhD"]
education_mapping = {level: number for number, level in enumerate(education_order)}

processed_base["parent_education_level"] = processed_base["parent_education"].map(education_mapping)

processed_base[["parent_education", "parent_education_level"]].head(10)

## 11. Scaling Numeric Features

Scaling puts numeric features on a similar range.

This matters for distance-based and gradient-based models, such as:

- Logistic Regression
- Linear Regression
- KNN
- SVM
- KMeans

Tree models, such as Random Forest, usually need less scaling.

In [ ]:
scaler = StandardScaler()
scaled_numeric = scaler.fit_transform(processed_base[numeric_columns])

scaled_numeric_df = pd.DataFrame(
    scaled_numeric,
    columns=[column + "_scaled" for column in numeric_columns],
)

scaled_numeric_df.head()

## 12. Train-Test Split Before Final Preprocessing

Important rule: split the data before fitting imputers, scalers, and encoders.

Why? Because the test set should behave like future unseen data. If preprocessing learns from the test set, that is called data leakage.

In [ ]:
# Select features and target.
# We do not use final_score as a feature because it directly determines passed.
feature_columns = [
    "study_hours",
    "attendance_rate",
    "previous_score",
    "sleep_hours",
    "internet_access",
    "parent_education",
    "extra_classes",
]

X = messy_data[feature_columns].copy()
y = messy_data["passed"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y,
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 13. A Reusable Cleaning Function

This function performs cleaning that should happen before scikit-learn preprocessing:

- Standardize category text
- Convert impossible numeric values to missing values
- Add engineered features

The imputation, encoding, and scaling will be handled by the pipeline after this.

In [ ]:
def clean_feature_table(df):
    df = df.copy()

    df["internet_access"] = df["internet_access"].apply(clean_yes_no)
    df["extra_classes"] = df["extra_classes"].apply(clean_yes_no)
    df["parent_education"] = df["parent_education"].apply(clean_parent_education)

    df.loc[(df["study_hours"] < 0) | (df["study_hours"] > 16), "study_hours"] = np.nan
    df.loc[(df["attendance_rate"] < 0) | (df["attendance_rate"] > 100), "attendance_rate"] = np.nan
    df.loc[(df["previous_score"] < 0) | (df["previous_score"] > 100), "previous_score"] = np.nan
    df.loc[(df["sleep_hours"] < 0) | (df["sleep_hours"] > 12), "sleep_hours"] = np.nan

    df["study_attendance_index"] = df["study_hours"] * (df["attendance_rate"] / 100)
    df["score_gap"] = 100 - df["previous_score"]
    df["healthy_sleep"] = np.where(df["sleep_hours"] >= 7, "Yes", "No")

    return df

X_train_clean = clean_feature_table(X_train)
X_test_clean = clean_feature_table(X_test)

X_train_clean.head()

## 14. Full Preprocessing Pipeline

A preprocessing pipeline makes the workflow reliable and repeatable.

Numeric columns:

1. Fill missing values with median.
2. Scale with StandardScaler.

Categorical columns:

1. Fill missing values with most frequent category.
2. One-hot encode categories.

Ordinal column:

1. Fill missing values.
2. Encode ordered education levels.

In [ ]:
numeric_features = [
    "study_hours",
    "attendance_rate",
    "previous_score",
    "sleep_hours",
    "study_attendance_index",
    "score_gap",
]

# We one-hot encode these because they do not have a numeric distance meaning.
nominal_features = ["internet_access", "extra_classes", "healthy_sleep"]

# We ordinal encode this because education level has a meaningful order.
ordinal_features = ["parent_education"]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# scikit-learn 1.2+ uses sparse_output. Older versions use sparse.
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

nominal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder),
    ]
)

ordinal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(categories=[["High School", "Bachelor", "Master", "PhD"]])),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("nominal", nominal_pipeline, nominal_features),
        ("ordinal", ordinal_pipeline, ordinal_features),
    ]
)

## 15. Transform Training and Test Data

Fit the preprocessor on training data only. Then transform both training and test data.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train_clean)
X_test_processed = preprocessor.transform(X_test_clean)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

In [ ]:
# Get readable column names after preprocessing.
processed_column_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=processed_column_names,
    index=X_train_clean.index,
)

X_train_processed_df.head()

## 16. Save Clean and Processed Data

Saving intermediate files is useful for teaching, debugging, and reproducibility.

In [ ]:
# Save manually cleaned data.
processed_base.to_csv("student_performance_cleaned_dataset.csv", index=False)

# Save model-ready training data.
model_ready_train = X_train_processed_df.copy()
model_ready_train["passed"] = y_train.values
model_ready_train.to_csv("student_performance_model_ready_train.csv", index=False)

print("Saved: student_performance_cleaned_dataset.csv")
print("Saved: student_performance_model_ready_train.csv")

## 17. Train a Model with the Preprocessing Pipeline

Instead of manually transforming the data first, we can combine preprocessing and modeling into one pipeline.

This is the recommended approach for most machine learning projects.

In [ ]:
# Clean first, then use a model pipeline for preprocessing plus classification.
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

model_pipeline.fit(X_train_clean, y_train)
predictions = model_pipeline.predict(X_test_clean)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification report:")
print(classification_report(y_test, predictions))

## 18. Process New Incoming Data

A real application receives new data. The same cleaning and preprocessing steps must be applied to new records before prediction.

In [ ]:
new_students = pd.DataFrame(
    {
        "study_hours": [8.5, -1, 4.0],
        "attendance_rate": [92, 130, 70],
        "previous_score": [81, 45, np.nan],
        "sleep_hours": [7.5, 18, 6.0],
        "internet_access": [" yes ", "NO", np.nan],
        "parent_education": ["bachelor", "High School", "Masters"],
        "extra_classes": ["Yes", "No ", " yes"],
    }
)

new_students_clean = clean_feature_table(new_students)
new_predictions = model_pipeline.predict(new_students_clean)

new_students_clean["predicted_passed"] = new_predictions
new_students_clean

## 19. Data Cleaning Checklist

Use this checklist for almost any tabular data science project:

1. Check rows and columns.
2. Check data types.
3. Check missing values.
4. Check duplicates.
5. Check impossible values.
6. Check category spelling and capitalization.
7. Check outliers.
8. Decide how to impute missing values.
9. Encode categorical variables.
10. Scale numeric features when the model needs it.
11. Split before fitting preprocessing tools.
12. Use pipelines to avoid data leakage.
13. Save cleaned and processed datasets when useful.
14. Document every cleaning decision.

## 20. Student Exercises

1. Add missing values to `final_score` and decide whether to fill or drop them.
2. Change the outlier rule from IQR capping to row removal. Compare row counts.
3. Try `drop_first=True` in one-hot encoding and compare the number of columns.
4. Use only one-hot encoding for `parent_education`. Compare with ordinal encoding.
5. Add a new feature called `high_attendance` for attendance above 85%.
6. Save the processed test set as a CSV.
7. Train the model without scaling and compare accuracy.
8. Explain what data leakage is using this notebook as an example.
9. Write a short cleaning report that lists every problem found in the messy dataset.
10. Create a reusable function that saves the data quality report to CSV.